# 🧪 시맨틱 캐싱 실습 (선택)

## 목적
유사한 프롬프트에 대해 **모델을 다시 호출하지 않고** 캐시에서 응답을 반환하는 시맨틱 캐싱을 검증합니다.

1. **동일 프롬프트 반복** → 캐시 히트 (지연 시간 급감)
2. **의미가 비슷한 프롬프트** → 유사도(≥0.8) 기반 히트
3. **전혀 다른 프롬프트** → 캐시 미스 (모델 신규 호출)

## 사전 조건 (중요)
이 노트북은 기본 배포에 **포함되지 않은 리소스**가 필요합니다. 다른 Lab 4 노트북과 달리 2단계 준비가 필요합니다:

**1단계 — 리소스 배포** (임베딩 모델 + Azure Redis Cache + APIM 외부 캐시 연결)
```bash
./scripts/deploy-semantic-caching.sh <suffix>
```

**2단계 — 정책 적용** (자동 적용 아님)
- Inbound: `azure-openai-semantic-cache-lookup` (score-threshold=0.8)
- Outbound: `azure-openai-semantic-cache-store` (duration=3600)
- 👉 구체적인 Portal 적용 단계와 붙여넣을 XML은 **아래 "사전 준비: 시맨틱 캐싱 정책 적용" 셀**에 있습니다.

> ⚠️ 위 2단계가 완료되지 않으면 캐시 히트가 발생하지 않고 **모든 호출이 미스로 동작**합니다.
> 그 경우 아래 테스트에서 지연 시간이 줄어들지 않으니, 정책 적용 여부를 먼저 확인하세요.


In [1]:
import os
import time
import requests
from dotenv import load_dotenv

load_dotenv("../../.env", override=True)

APIM_URL = os.getenv("APIM_URL")
SUBSCRIPTION_KEY = os.getenv("APIM_SUBSCRIPTION_KEY")
DEPLOYMENT_NAME = os.getenv("DEPLOYMENT_NAME", "gpt-4.1-nano")
API_VERSION = "2025-04-01-preview"

assert APIM_URL, "❌ APIM_URL이 설정되지 않았습니다. .env를 확인하세요."
assert SUBSCRIPTION_KEY, "❌ APIM_SUBSCRIPTION_KEY가 설정되지 않았습니다. .env를 확인하세요."

BASE_URL = f"{APIM_URL}/openai/deployments/{DEPLOYMENT_NAME}/chat/completions"
HEADERS = {
    "Content-Type": "application/json",
    "Ocp-Apim-Subscription-Key": SUBSCRIPTION_KEY,
}


def call_chat(prompt, max_tokens=100, temperature=0.0):
    """APIM → Azure OpenAI Chat Completion 호출 후 (응답, 지연 ms) 반환."""
    start = time.time()
    resp = requests.post(
        BASE_URL,
        params={"api-version": API_VERSION},
        headers=HEADERS,
        json={
            "messages": [{"role": "user", "content": prompt}],
            "max_tokens": max_tokens,
            "temperature": temperature,
        },
        timeout=60,
    )
    ms = int((time.time() - start) * 1000)
    return resp, ms


print("✅ 환경 설정 완료")
print(f"   APIM URL:   {APIM_URL}")
print(f"   Deployment: {DEPLOYMENT_NAME}")
print("\n💡 캐시 히트 판단 기준: 동일/유사 프롬프트의 2차 호출 지연이 크게 줄어들면 히트입니다.")
print("   (시맨틱 캐시는 별도 헤더를 남기지 않으므로 지연 시간으로 판단합니다.)")


✅ 환경 설정 완료
   APIM URL:   https://apim-ai-gw-aigateway-20260716.azure-api.net
   Deployment: gpt-4.1-nano

💡 캐시 히트 판단 기준: 동일/유사 프롬프트의 2차 호출 지연이 크게 줄어들면 히트입니다.
   (시맨틱 캐시는 별도 헤더를 남기지 않으므로 지연 시간으로 판단합니다.)


---
## 사전 준비: 시맨틱 캐싱 정책 적용 (Portal)

> ⚠️ **아래 테스트를 실행하기 전에 반드시 이 정책을 먼저 적용하세요.**
> 적용하지 않으면 캐시가 동작하지 않아 모든 호출이 미스로 나옵니다.

### 진행 순서

1. 먼저 리소스 배포가 끝났는지 확인 (임베딩 모델 + Redis + 외부 캐시 연결):
   ```bash
   ./scripts/deploy-semantic-caching.sh <suffix>
   ```
2. Azure Portal → APIM → **APIs** → **Azure OpenAI** → **All operations**
3. **Inbound processing** 영역의 **</>** 클릭 (Code View)
4. `<base />` 바로 아래에 **cache-lookup** 을 추가:

```xml
<!-- Inbound processing -->
<azure-openai-semantic-cache-lookup
    score-threshold="0.8"
    embeddings-backend-id="embedding-backend"
    embeddings-backend-auth="system-assigned" />
```

5. **Outbound processing** 영역의 **</>** 클릭 → `<base />` 아래에 **cache-store** 를 추가 → **Save**:

```xml
<!-- Outbound processing -->
<azure-openai-semantic-cache-store duration="3600" />
```

> 💡 **왜 둘 다 필요한가**
> - `cache-lookup`(inbound): 요청이 들어올 때 임베딩 유사도로 캐시를 조회 → 히트면 백엔드 호출 없이 즉시 응답
> - `cache-store`(outbound): 백엔드 응답을 캐시에 저장 → 다음 유사 요청이 히트하도록
>
> `embeddings-backend-id="embedding-backend"`는 배포 스크립트가 등록한 임베딩 백엔드 이름이며,
> `embeddings-backend-auth="system-assigned"`는 APIM의 Managed Identity로 임베딩 모델을 호출한다는 의미입니다.

### 🔄 테스트 후 정책 제거 (선택)
실습이 끝나면 위에서 추가한 `azure-openai-semantic-cache-lookup`(inbound)과 `azure-openai-semantic-cache-store`(outbound) 블록을 삭제하고 **Save** 하세요.


---
## Test 1: 동일 프롬프트 반복 (캐시 미스 → 캐시 히트)

같은 프롬프트를 두 번 호출합니다.
- **1차 호출**: 캐시에 없으므로 모델을 호출 → 느림 (캐시 미스)
- **2차 호출**: 캐시에서 반환 → 빠름 (캐시 히트)


In [ ]:
print("▶ Test 1: 동일 프롬프트 반복\n")

prompt = "Azure API Management의 핵심 기능 3가지를 한 문장으로 설명해줘."

# 1차 호출 — 캐시 미스 (모델 호출)
resp1, ms1 = call_chat(prompt)
if resp1.status_code == 200:
    ans1 = resp1.json()["choices"][0]["message"]["content"].strip()[:80]
else:
    ans1 = resp1.text[:100]
print(f"  1차 호출 (캐시 미스): HTTP {resp1.status_code}, {ms1}ms")
print(f"    → {ans1}\n")

# 캐시 저장(outbound)이 완료될 시간을 잠시 대기
time.sleep(2)

# 2차 호출 — 동일 프롬프트 (캐시 히트 기대)
resp2, ms2 = call_chat(prompt)
if resp2.status_code == 200:
    ans2 = resp2.json()["choices"][0]["message"]["content"].strip()[:80]
else:
    ans2 = resp2.text[:100]
print(f"  2차 호출 (캐시 히트 기대): HTTP {resp2.status_code}, {ms2}ms")
print(f"    → {ans2}\n")

if resp1.status_code == 200 and resp2.status_code == 200:
    speedup = ms1 / ms2 if ms2 > 0 else 0
    print(f"  ⏱️  지연 비교: {ms1}ms → {ms2}ms  ({speedup:.1f}배)")
    if ms2 < ms1 * 0.5:
        print("  ✅ 2차 호출이 크게 빨라짐 → 캐시 히트로 판단")
    else:
        print("  ⚠️ 지연 차이가 작습니다. 캐시 정책(lookup/store)이 적용되었는지 확인하세요.")


---
## Test 2: 의미가 비슷한 프롬프트 (시맨틱 히트)

표현은 다르지만 **의미가 유사한** 프롬프트를 호출합니다.
일반 캐시(정확한 문자열 매칭)라면 미스지만, 시맨틱 캐시는 임베딩 **유사도가 0.8 이상**이면 히트합니다.


In [ ]:
print("▶ Test 2: 의미가 비슷한 프롬프트\n")

# Test 1의 prompt와 표현은 다르지만 의미가 유사
similar_prompt = "APIM의 주요 기능 세 가지를 간단히 알려줘."

resp, ms = call_chat(similar_prompt)
if resp.status_code == 200:
    ans = resp.json()["choices"][0]["message"]["content"].strip()[:80]
else:
    ans = resp.text[:100]
print(f"  유사 프롬프트: HTTP {resp.status_code}, {ms}ms")
print(f"    → {ans}\n")

if resp.status_code == 200:
    if ms < ms1 * 0.5:
        print("  ✅ 표현이 달라도 빠름 → 유사도 기반(≥0.8) 시맨틱 캐시 히트")
    else:
        print("  ℹ️ 미스일 수 있습니다. 유사도가 score-threshold(0.8) 미만이면 미스입니다.")
        print("     더 비슷하게 바꾸거나, 정책의 score-threshold를 낮춰 테스트해 보세요.")


---
## Test 3: 전혀 다른 프롬프트 (캐시 미스)

의미가 다른 프롬프트는 캐시에 없으므로 모델을 새로 호출합니다 → 다시 느려져야 정상입니다.


In [ ]:
print("▶ Test 3: 전혀 다른 프롬프트\n")

different_prompt = "피보나치 수열을 계산하는 파이썬 함수를 작성해줘."

resp, ms = call_chat(different_prompt)
if resp.status_code == 200:
    ans = resp.json()["choices"][0]["message"]["content"].strip()[:80]
else:
    ans = resp.text[:100]
print(f"  다른 프롬프트: HTTP {resp.status_code}, {ms}ms")
print(f"    → {ans}\n")

if resp.status_code == 200:
    if ms > ms2 * 2:
        print("  ✅ 다시 느려짐 → 캐시 미스(모델 신규 호출)로 판단")
    else:
        print("  ℹ️ 예상보다 빠릅니다. 우연히 캐시된 유사 프롬프트가 있을 수 있습니다.")


---
## ✅ 체크리스트

| # | 확인 항목 | 기대 결과 |
|---|-----------|----------|
| 1 | 동일 프롬프트 2차 호출 | 1차보다 크게 빠름 (캐시 히트) |
| 2 | 의미가 비슷한 프롬프트 | 표현이 달라도 빠름 (유사도 히트) |
| 3 | 전혀 다른 프롬프트 | 다시 느려짐 (캐시 미스) |

## 💡 시맨틱 캐싱의 이점
- **비용 절감**: 캐시 히트 시 모델을 호출하지 않아 토큰 비용이 발생하지 않음
- **지연 단축**: 임베딩 조회 + Redis 반환만 수행하므로 응답이 훨씬 빠름
- **유사도 매칭**: 정확히 같은 문장이 아니어도(오타·어순·표현 차이) 히트 가능

> ⚠️ **주의**
> - `score-threshold`를 너무 낮추면(예: 0.5) 의미가 다른 프롬프트도 히트되어 **잘못된 응답**을 반환할 수 있습니다.
> - 캐시는 `duration`(기본 3600초) 동안 유지되므로, 최신성이 중요한 응답에는 캐싱을 피하세요.
> - 실습 후 비용이 걱정되면 `./scripts/cleanup.sh`로 리소스를 정리하세요.
